In [6]:
!pip install -q transformers datasets evaluate accelerate

In [7]:
import torch
import json
import os

from google.colab import drive

drive.mount("/content/drive")
PROJECT_DIR = "/content/drive/MyDrive/ShweMyanmar"
DATA_DIR = f"{PROJECT_DIR}/data"
MODEL_DIR = f"{PROJECT_DIR}/models"
RESULTS_DIR = f"{PROJECT_DIR}/results"

TRAIN_PATH = f"{DATA_DIR}/pos_sample_300.jsonl"
VAL_PATH = f"{DATA_DIR}/pos_sample_100.jsonl"
TEST_PATH = f"{DATA_DIR}/pos_sample_100.jsonl"


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
from transformers import AutoTokenizer
MODEL_NAME = "bert-base-multilingual-cased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

In [8]:
with open(TRAIN_PATH, "r", encoding="utf-8") as f:
    for i in range(5):
        line = f.readline()
        print(f"LINE {i+1}:")
        print(line[:500])
        print("-" * 50)

LINE 1:
{"id": "sent_23222", "text": "ခင်ဗျားကိုယ့်ကိုကိုယ်ချည်းပဲအပစ်မတင်စမ်းပါနဲ့။", "tokens": ["ခင်ဗျား", "ကိုယ့်", "ကို", "ကိုယ်", "ချည်း", "ပဲ", "အပစ်မတင်", "စမ်း", "ပါ", "နဲ့", "။"], "pos_tags": ["pron", "pron", "ppm", "pron", "part", "part", "v", "part", "part", "part", "punc"], "pos_ids": [10, 10, 9, 10, 8, 8, 14, 8, 8, 8, 11]}

--------------------------------------------------
LINE 2:
{"id": "sent_18111", "text": "ဇိမ်ယူတတ်တဲ့အကျင့်ကိုမပြင်ရင်တော့အောင်မြင်ဖို့မလွယ်ဘူး။", "tokens": ["ဇိမ်ယူ", "တတ်", "တဲ့", "အကျင့်", "ကို", "မ", "ပြင်", "ရင်", "တော့", "အောင်မြင်", "ဖို့", "မ", "လွယ်", "ဘူး", "။"], "pos_tags": ["v", "part", "part", "n", "ppm", "part", "v", "conj", "part", "v", "part", "part", "v", "part", "punc"], "pos_ids": [14, 8, 8, 6, 9, 8, 14, 3, 8, 14, 8, 8, 14, 8, 11]}

--------------------------------------------------
LINE 3:
{"id": "sent_41139", "text": "ဒေသရဲ့ခရီးဝေးလမ်းလျှောက်လမ်းကြောင်းတွေနဲ့မြေပုံကိုကျွန်တော်ဘယ်နေရာမှာကြည့်နိုင်မလဲ။", "tokens": ["ဒေသ", "ရဲ့", "ခရီး

In [9]:
def load_jsonl(path):
    data = []

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            if line:
                data.append(json.loads(line))

    return data

train_data = load_jsonl(TRAIN_PATH)
val_data = load_jsonl(VAL_PATH)
test_data = load_jsonl(TEST_PATH)

print("Train:", len(train_data))
print("Validation:", len(val_data))
print("Test:", len(test_data))

Train: 300
Validation: 100
Test: 100


In [10]:
pos_labels = sorted(
    {
        tag
        for example in train_data
        for tag in example["pos_tags"]
    }
)

print(pos_labels)
print("Number of POS classes:", len(pos_labels))

['abb', 'adj', 'adv', 'conj', 'fw', 'int', 'n', 'num', 'part', 'ppm', 'pron', 'punc', 'tn', 'v']
Number of POS classes: 14


In [11]:
label2id = {
    label: i
    for i, label in enumerate(pos_labels)
}

id2label = {
    i: label
    for label, i in label2id.items()
}

print(label2id)

{'abb': 0, 'adj': 1, 'adv': 2, 'conj': 3, 'fw': 4, 'int': 5, 'n': 6, 'num': 7, 'part': 8, 'ppm': 9, 'pron': 10, 'punc': 11, 'tn': 12, 'v': 13}


In [12]:
example = train_data[0]

words = example["tokens"]

encoding = tokenizer(
    words,
    is_split_into_words=True,
    truncation=True,
    max_length=128
)

In [13]:
bert_tokens = tokenizer.convert_ids_to_tokens(
    encoding["input_ids"]
)

print(bert_tokens)
print(encoding.word_ids())

['[CLS]', 'ခ', '##င်', '##ဗ', '##ျား', 'ကို', '##ယ', '##့်', 'ကို', 'ကို', '##ယ်', 'ခ', '##ျ', '##ည်း', 'ပ', '##ဲ', 'အ', '##ပ', '##စ်', '##မ', '##တ', '##င်', 'စ', '##မ်း', 'ပ', '##ါ', 'န', '##ဲ့', '။', '[SEP]']
[None, 0, 0, 0, 0, 1, 1, 1, 2, 3, 3, 4, 4, 4, 5, 5, 6, 6, 6, 6, 6, 6, 7, 7, 8, 8, 9, 9, 10, None]


In [14]:
def tokenize_and_align_labels(example):
    tokenized = tokenizer(
        example["tokens"],
        truncation=True,
        is_split_into_words=True,
        max_length=128
    )

    word_ids = tokenized.word_ids()

    labels = []
    previous_word_id = None

    for word_id in word_ids:

        # [CLS], [SEP], etc.
        if word_id is None:
            labels.append(-100)

        # First BERT piece of a word
        elif word_id != previous_word_id:
            pos_tag = example["pos_tags"][word_id]
            labels.append(label2id[pos_tag])

        # Remaining pieces of the same word
        else:
            labels.append(-100)

        previous_word_id = word_id

    tokenized["labels"] = labels

    return tokenized

In [15]:
example = train_data[0]

encoded = tokenize_and_align_labels(example)

tokens = tokenizer.convert_ids_to_tokens(
    encoded["input_ids"]
)

labels = encoded["labels"]

for token, label in zip(tokens, labels):

    if label == -100:
        readable_label = "IGNORE"
    else:
        readable_label = id2label[label]

    print(f"{token:20} {readable_label}")

[CLS]                IGNORE
ခ                    pron
##င်                 IGNORE
##ဗ                  IGNORE
##ျား                IGNORE
ကို                  pron
##ယ                  IGNORE
##့်                 IGNORE
ကို                  ppm
ကို                  pron
##ယ်                 IGNORE
ခ                    part
##ျ                  IGNORE
##ည်း                IGNORE
ပ                    part
##ဲ                  IGNORE
အ                    v
##ပ                  IGNORE
##စ်                 IGNORE
##မ                  IGNORE
##တ                  IGNORE
##င်                 IGNORE
စ                    part
##မ်း                IGNORE
ပ                    part
##ါ                  IGNORE
န                    part
##ဲ့                 IGNORE
။                    punc
[SEP]                IGNORE


In [16]:
from datasets import Dataset

train_dataset = Dataset.from_list(train_data)
val_dataset = Dataset.from_list(val_data)
test_dataset = Dataset.from_list(test_data)

In [17]:
tokenized_train = train_dataset.map(
    tokenize_and_align_labels
)

tokenized_val = val_dataset.map(
    tokenize_and_align_labels
)

tokenized_test = test_dataset.map(
    tokenize_and_align_labels
)

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

In [18]:
print(tokenized_train[0])

{'id': 'sent_23222', 'text': 'ခင်ဗျားကိုယ့်ကိုကိုယ်ချည်းပဲအပစ်မတင်စမ်းပါနဲ့။', 'tokens': ['ခင်ဗျား', 'ကိုယ့်', 'ကို', 'ကိုယ်', 'ချည်း', 'ပဲ', 'အပစ်မတင်', 'စမ်း', 'ပါ', 'နဲ့', '။'], 'pos_tags': ['pron', 'pron', 'ppm', 'pron', 'part', 'part', 'v', 'part', 'part', 'part', 'punc'], 'pos_ids': [10, 10, 9, 10, 8, 8, 14, 8, 8, 8, 11], 'input_ids': [101, 1493, 28235, 111491, 92751, 39194, 105942, 74024, 39194, 39194, 39815, 1493, 111503, 92803, 1512, 110430, 1524, 111489, 43851, 64529, 111484, 28235, 1497, 78278, 1512, 50428, 1511, 89114, 1559, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'labels': [-100, 10, -100, -100, -100, 10, -100, -100, 9, 10, -100, 8, -100, -100, 8, -100, 13, -100, -100, -100, -100, -100, 8, -100, 8, -100, 8, -100, 11, -100]}


In [19]:
from transformers import AutoModelForTokenClassification
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(pos_labels),
    id2label=id2label,
    label2id=label2id
)

model.safetensors: reconstructing file:   0%|          |  0.00B /  714MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params

In [21]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model.to(device)

BertForTokenClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(119547, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, e

In [24]:
!pip install scikit-learn

In [25]:
import numpy as np

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support
)

In [26]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    # Model outputs scores for every POS class.
    # Choose the class with the highest score.
    predictions = np.argmax(predictions, axis=-1)

    true_predictions = []
    true_labels = []

    for prediction, label in zip(predictions, labels):
        for pred_id, label_id in zip(prediction, label):

            # Ignore special tokens and ignored subword pieces
            if label_id != -100:
                true_predictions.append(int(pred_id))
                true_labels.append(int(label_id))

    accuracy = accuracy_score(
        true_labels,
        true_predictions
    )

    precision, recall, f1, _ = precision_recall_fscore_support(
        true_labels,
        true_predictions,
        average="macro",
        zero_division=0
    )

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

In [27]:
from transformers import DataCollatorForTokenClassification
data_collator = DataCollatorForTokenClassification(
    tokenizer=tokenizer
)

In [28]:
from transformers import TrainingArguments
training_args = TrainingArguments(
    output_dir=f"{MODEL_DIR}/pos_checkpoints",

    learning_rate=2e-5,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    num_train_epochs=3,

    weight_decay=0.01,

    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,

    metric_for_best_model="f1",
    greater_is_better=True,

    logging_steps=20,

    report_to="none"
)

In [29]:
import transformers
print(transformers.__version__)
from transformers import Trainer
trainer = Trainer(
    model=model,
    args=training_args,

    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,

    processing_class=tokenizer,

    data_collator=data_collator,

    compute_metrics=compute_metrics
)

5.16.1


In [30]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,1.857150,0.931430,0.736498,0.368618,0.339030,0.318726
2,0.862430,0.690016,0.798691,0.501921,0.419983,0.428067
3,0.686099,0.627695,0.815057,0.632501,0.455110,0.465297


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

TrainOutput(global_step=114, training_loss=1.022572032192297, metrics={'train_runtime': 74.3902, 'train_samples_per_second': 12.098, 'train_steps_per_second': 1.532, 'total_flos': 40652386670208.0, 'train_loss': 1.022572032192297, 'epoch': 3.0})

In [31]:
test_results = trainer.evaluate(
    tokenized_test
)

print(test_results)

Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.686099,0.627695,3,0.815057,0.632501,0.455110,0.465297


{'eval_loss': 0.6276949644088745, 'eval_accuracy': 0.8150572831423896, 'eval_precision': 0.6325009514679985, 'eval_recall': 0.45511042492569764, 'eval_f1': 0.46529665153503735}


In [32]:
FINAL_MODEL_DIR = f"{MODEL_DIR}/pos"

trainer.save_model(FINAL_MODEL_DIR)

tokenizer.save_pretrained(FINAL_MODEL_DIR)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/drive/MyDrive/ShweMyanmar/models/pos/tokenizer_config.json',
 '/content/drive/MyDrive/ShweMyanmar/models/pos/tokenizer.json')

In [33]:
with open(
    f"{FINAL_MODEL_DIR}/label2id.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        label2id,
        f,
        ensure_ascii=False,
        indent=2
    )

In [34]:
print("Number of training examples:", len(tokenized_train))
print("Number of validation examples:", len(tokenized_val))
print("Number of test examples:", len(tokenized_test))

print("\nPOS Labels:")
print(label2id)

print("\nTest Results:")
print(test_results)

Number of training examples: 300
Number of validation examples: 100
Number of test examples: 100

POS Labels:
{'abb': 0, 'adj': 1, 'adv': 2, 'conj': 3, 'fw': 4, 'int': 5, 'n': 6, 'num': 7, 'part': 8, 'ppm': 9, 'pron': 10, 'punc': 11, 'tn': 12, 'v': 13}

Test Results:
{'eval_loss': 0.6276949644088745, 'eval_accuracy': 0.8150572831423896, 'eval_precision': 0.6325009514679985, 'eval_recall': 0.45511042492569764, 'eval_f1': 0.46529665153503735}


In [35]:
import json
import os

os.makedirs(RESULTS_DIR, exist_ok=True)

with open(
    f"{RESULTS_DIR}/pos_sample_300_results.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        {
            "model": MODEL_NAME,
            "dataset": "300-line sample",
            "num_train": len(tokenized_train),
            "num_validation": len(tokenized_val),
            "num_test": len(tokenized_test),
            "labels": label2id,
            "test_results": {
                k: float(v) if hasattr(v, "__float__") else v
                for k, v in test_results.items()
            }
        },
        f,
        ensure_ascii=False,
        indent=2
    )

print("Results saved.")

Results saved.


In [36]:
from transformers import AutoTokenizer, AutoModelForTokenClassification

FINAL_MODEL_DIR = f"{MODEL_DIR}/pos"

test_tokenizer = AutoTokenizer.from_pretrained(
    FINAL_MODEL_DIR
)

test_model = AutoModelForTokenClassification.from_pretrained(
    FINAL_MODEL_DIR
)

print("Saved POS model loaded successfully!")
print("Number of labels:", test_model.config.num_labels)
print(test_model.config.id2label)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Saved POS model loaded successfully!
Number of labels: 14
{0: 'abb', 1: 'adj', 2: 'adv', 3: 'conj', 4: 'fw', 5: 'int', 6: 'n', 7: 'num', 8: 'part', 9: 'ppm', 10: 'pron', 11: 'punc', 12: 'tn', 13: 'v'}
